In [1]:
import pandas as pd
import os
import glob

# Month Wise PO's Read

In [2]:
import os
import pandas as pd

folder_path = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\Quick_Commerece_Input_ALL\Instamart\Month_Wise_PO's"

dfs = []

for file in os.listdir(folder_path):
    if file.lower().endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        print(f"📥 Reading: {file}")
        temp_df = pd.read_csv(file_path)
        dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print(f"✅ Combined DataFrame shape: {df.shape}")


📥 Reading: 10JAN_TO_8FEB.csv
📥 Reading: LAST 30 DAYS TILL 10 JUNE  2026.csv
📥 Reading: LAST 30 DAYS TILL 11 JUNE  2026.csv
📥 Reading: LAST 30 DAYS TILL 3 JUNE  2026.csv
📥 Reading: LAST 60 DAYS TILL 17 APRIL 2026.csv
📥 Reading: LAST 60 DAYS TILL 23 APRIL 2026.csv
📥 Reading: LAST 60 DAYS TILL 24 APRIL 2026.csv
📥 Reading: LAST 60 DAYS TILL 25 MAY 2026.csv
📥 Reading: LAST 60 DAYS TILL 29 APRIL 2026.csv
📥 Reading: last 60 days till 3 feb.csv
📥 Reading: last 60 days till 7 MAY.csv
📥 Reading: LAST_3-DAYS_TILL_5FEB.csv
📥 Reading: LAST_30DAYS_TILL_16FEB.csv
📥 Reading: LAST_30DAYS_TILL_19MARCH.csv
📥 Reading: LAST_30_DAYS_TILL_1april.csv
📥 Reading: LAST_60-DAYS_TILL_11MAY.csv
📥 Reading: LAST_60-DAYS_TILL_20 MAY.csv
📥 Reading: LAST_60-DAYS_TILL_24FEB.csv
📥 Reading: LAST_60-DAYS_TILL_26FEB.csv
📥 Reading: LAST_60-DAYS_TILL_2june.csv
📥 Reading: LAST_60DAYS_TILL_23FEB.csv
📥 Reading: LAST_60DAYS_TILL_6MARCH.csv
📥 Reading: LAST_60DAYS_TILL_9MARCH.csv
📥 Reading: last_60_days_till_19feb.csv
📥 Reading: LAS

In [3]:
import pandas as pd

# ===============================
# STEP 1: Ensure PoModifiedAt is datetime
# ===============================
df["PoModifiedAt"] = pd.to_datetime(
    df["PoModifiedAt"],
    errors="coerce"
)

# ===============================
# STEP 2: Sort so latest comes first
# ===============================
df = df.sort_values(
    by=["PoNumber", "SkuCode", "PoModifiedAt"],
    ascending=[True, True, False]
)

# ===============================
# STEP 3: Drop duplicates (keep latest)
# ===============================
df = df.drop_duplicates(
    subset=["PoNumber", "SkuCode"],
    keep="first"
).reset_index(drop=True)

print(f"✅ Final rows after deduplication: {df.shape[0]}")


✅ Final rows after deduplication: 1768


In [4]:
df['PoLineValueWithTax'].sum()

np.float64(10681964.78)

## Rename and take only required columns in PO


In [5]:
import pandas as pd

# Step 1: Rename columns
df = df.rename(columns={
    'Mrp':'PO MRP',
    'PoNumber': 'PO Number',
    'PoCreatedAt': 'PO Creation Date',
    'VendorName': 'PO Delivered By Address',
    'Entity': 'PO Delivered To',
    'SkuCode': 'PO Item Code',
    'SkuDescription': 'PO Product Description',
    'UnitBasedCost': 'PO Basic Cost Price',
    'Tax': 'PO Tax Amt',
    'OrderedQty': 'PO_quantity',
    'PoLineValueWithTax': 'PO_Amount',
    'PoExpiryDate': 'PO Expiry Date'
})

# Step 2: Select only required columns
required_columns = [
    'PO Number',
    'PO Creation Date',
    'PO Expiry Date',
    'PO Delivered By Address',
    'PO Delivered To',
    'PO Item Code',
    'PO Product Description',
    'PO Basic Cost Price',
    'PO Tax Amt',
    'PO_quantity',
    'PO_Amount',
    'PO MRP'
]

df = df[required_columns]
df['HSN Code'] = ""
df['PO Product UPC'] = ""
df['PO Landing Rate'] = ""

### Change the format of PO Creation Date and PO Amount

In [6]:
import pandas as pd

# Convert PO Creation Date to datetime and keep only date
df['PO Creation Date'] = pd.to_datetime(
    df['PO Creation Date'],
    errors='coerce'
).dt.date

# Convert PO_Amount to numeric and round to 2 decimals
df['PO_Amount'] = pd.to_numeric(df['PO_Amount'], errors='coerce').round(2)
df["PO Item Code"] = df["PO Item Code"].astype(str).str.strip()


# Add SKU details

In [7]:
import pandas as pd

# ---- Read helper file ----
helper_file = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\helper_file.xlsx"

helper_df = pd.read_excel(helper_file, dtype=str)

# ---- Clean column names & key columns ----
helper_df.columns = helper_df.columns.str.strip()
helper_df["Item Code"] = helper_df["Item Code"].astype(str).str.strip()
helper_df["SKU"] = helper_df["SKU"].astype(str).str.strip()

# ---- Clean PO Item Code in main df ----
df["PO Item Code"] = df["PO Item Code"].astype(str).str.strip()

# ---- Create mapping dictionaries ----
itemcode_to_sku = (
    helper_df
    .dropna(subset=["Item Code", "SKU"])
    .set_index("Item Code")["SKU"]
    .to_dict()
)

sku_to_category = helper_df.set_index("SKU")["Category"].to_dict()
sku_to_material = helper_df.set_index("SKU")["Material"].to_dict()
sku_to_subcategory = helper_df.set_index("SKU")["Sub Category"].to_dict()
sku_to_box_case = helper_df.set_index("SKU")["Box/Case"].to_dict()

# ---- Map values to df ----
df["SKU"] = df["PO Item Code"].map(itemcode_to_sku)

df["Category"] = df["SKU"].map(sku_to_category)
df["Material"] = df["SKU"].map(sku_to_material)
df["Sub Category"] = df["SKU"].map(sku_to_subcategory)
df["Box/Case"] = df["SKU"].map(sku_to_box_case)

# ---- Find missing item codes ----
missing_item_codes = (
    df.loc[df["SKU"].isna(), "PO Item Code"]
    .unique()
    .tolist()
)

missing_item_codes

[]

In [8]:
df_po_month=df.copy()

In [9]:
is_unique = not df_po_month.duplicated(subset=['PO Number', 'SKU']).any()

print("Is PO No + SKU combination unique?", is_unique)

Is PO No + SKU combination unique? True


In [10]:
df_po_month

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO_quantity,PO_Amount,PO MRP,HSN Code,PO Product UPC,PO Landing Rate,SKU,Category,Material,Sub Category,Box/Case
0,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,203624,"Eco Soul 180ml Round, Disposable Bowl 25.0 Pieces",51.00,326.40,128,6854.4,119.0,,,,CBB6RO25UB,Home Essentials,Bagasse,Kitchen Essentials,64
1,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,284873,"Eco Soul 9 Round, Eco Soul Disposable Plate 10...",70.95,532.14,150,11175.0,149.0,,,,CBP9RO10UB,Home Essentials,Bagasse,Tableware,50
2,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,394372,"Eco Soul 1Ply, 100Count, Disposable Paper Napk...",41.48,2240.08,300,14685.0,89.0,,,,CBDN1P100P01,Home Essentials,Bagasse,Kitchen Essentials,50
3,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,428727,"Eco Soul 140mm, Disposable Spoon 50.0 Pieces",47.14,707.14,300,14850.0,99.0,,,,BS140MM50,Home Essentials,Bagasse,Tableware,100
4,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,756550,"Eco Soul 12 Round, Eco Soul Palm Leaf Plate 10...",135.11,0.00,150,20266.5,229.0,,,,PLP12RO10,Home Essentials,Palm Leaf,Tableware,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,944680,"Eco Soul 11Round,Eco Soul 4CP Disposable Plate...",94.76,170.57,36,3582.0,199.0,,,,CBP11RO4C10UB,Home Essentials,Bagasse,Tableware,36
1764,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,946960,"Eco Soul 6 Square, Eco Soul Palm Leaf Plate 10...",76.11,0.00,20,1522.2,129.0,,,,PLP6SQ10,Home Essentials,Palm Leaf,Tableware,20
1765,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,963761,"Eco Soul 250ml, Multi Colour Paper Cup 25.0 Pi...",60.13,432.92,40,2838.0,129.0,,,,CPHC8OZ25NL,Home Essentials,Paper,Drinkware,40
1766,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,973525,Eco Soul Beer Pong Glasses | Red | 16 Oz 1.0 Pack,59.03,424.98,40,2786.0,199.0,,,,OBGR16OZ10,Home Essentials,Plastic,Tableware,40


# Month Wise Invoice Read

In [11]:
import os
import pandas as pd

folder_path = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\Quick_Commerece_Input_ALL\All_invoices"

dfs = []

for file in os.listdir(folder_path):
    if file.lower().endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        print(f"📥 Reading: {file}")
        temp_df = pd.read_csv(file_path)
        dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print(f"✅ Combined DataFrame shape: {df.shape}")


📥 Reading: 1DEC_TO_FEB.csv


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_16764\1186993499.py:12: DtypeWarning: Columns (74,87,89,98,101,126,144,145,170,171,172,173,180) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


📥 Reading: 1feb_to_april.csv


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_16764\1186993499.py:12: DtypeWarning: Columns (44,87,89,95,98,144,145,170,171,172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


📥 Reading: 1JAN_TO_MARCH.csv


...

📥 Reading: May25-july25.csv


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_16764\1186993499.py:12: DtypeWarning: Columns (26,44,55,74,82,84,88,90,96,115,116,117,118,121,123,124,126,130,132,134,135,136,138,139,140,141,153,155,158,172,173,174,176,177,179,182,183) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


📥 Reading: Nov24-jan25.csv
📥 Reading: Nov25-jan26.csv


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_16764\1186993499.py:12: DtypeWarning: Columns (26,44,55,74,82,84,88,90,93,96,115,116,117,118,121,123,124,126,132,134,135,136,138,139,140,141,158,167,172,173,174,175,182) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


✅ Combined DataFrame shape: (166083, 192)


In [12]:
df

,Invoice Date,Invoice ID,Invoice Number,Invoice Status,Invoice Type,Customer Name,Customer ID,Customer Number,Location ID,Location Name,...,CF.Dispatch From,CF.Shipping Charges,CF.Bookmark this invoice,LINEITEM.TAG.Branch,LINEITEM.TAG.Branch-Maharashtra,LINEITEM.TAG.Customer,LINEITEM.TAG.Geography,LINEITEM.TAG.Location,TAG.MONTHLY EXP,TAG.TAG
0,2026-01-01,1.051890e+18,FK/2025-26/094,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-01,1.051890e+18,FK/2025-26/094,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-01-01,1.051890e+18,FK/2025-26/094,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-01-01,1.051890e+18,FK/2025-26/095,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-01-01,1.051890e+18,FK/2025-26/096,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166078,2026-01-31,1.051890e+18,Amazon/245,Overdue,Invoice,Amazon Seller Services Pvt Ltd,1.051890e+18,CUS-00003,1.051890e+18,Branch-Telangana,...,Amazon Inventory Warehouse Telangana,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166079,2026-01-31,1.051890e+18,Amazon/245,Overdue,Invoice,Amazon Seller Services Pvt Ltd,1.051890e+18,CUS-00003,1.051890e+18,Branch-Telangana,...,Amazon Inventory Warehouse Telangana,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166080,2026-01-31,1.051890e+18,Amazon/245,Overdue,Invoice,Amazon Seller Services Pvt Ltd,1.051890e+18,CUS-00003,1.051890e+18,Branch-Telangana,...,Amazon Inventory Warehouse Telangana,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166081,2026-01-31,1.051890e+18,Amazon/245,Overdue,Invoice,Amazon Seller Services Pvt Ltd,1.051890e+18,CUS-00003,1.051890e+18,Branch-Telangana,...,Amazon Inventory Warehouse Telangana,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Select only instamart invoice

In [13]:
df = df[
    df['Customer Name']
    .astype(str)
    .str.strip()
    .str.upper()
    .eq('SCOOTSY LOGISTICS PRIVATE LIMITED')
]


In [14]:
df

,Invoice Date,Invoice ID,Invoice Number,Invoice Status,Invoice Type,Customer Name,Customer ID,Customer Number,Location ID,Location Name,...,CF.Dispatch From,CF.Shipping Charges,CF.Bookmark this invoice,LINEITEM.TAG.Branch,LINEITEM.TAG.Branch-Maharashtra,LINEITEM.TAG.Customer,LINEITEM.TAG.Geography,LINEITEM.TAG.Location,TAG.MONTHLY EXP,TAG.TAG
547,2025-12-10,1.051890e+18,IN/2025-26/001,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Bangalore,...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
548,2025-12-10,1.051890e+18,IN/2025-26/001,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Bangalore,...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
549,2025-12-10,1.051890e+18,IN/2025-26/001,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Bangalore,...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
550,2025-12-10,1.051890e+18,IN/2025-26/001,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Bangalore,...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
551,2025-12-10,1.051890e+18,IN/2025-26/001,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Bangalore,...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151018,2026-01-27,1.051890e+18,IN/2025-26/064,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Maharashtra,...,"Waredepot, C/O Top India Logistics, Gala No. 1...",NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151019,2026-01-27,1.051890e+18,IN/2025-26/064,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Maharashtra,...,"Waredepot, C/O Top India Logistics, Gala No. 1...",NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151020,2026-01-27,1.051890e+18,IN/2025-26/064,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Maharashtra,...,"Waredepot, C/O Top India Logistics, Gala No. 1...",NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151021,2026-01-27,1.051890e+18,IN/2025-26/064,Closed,Invoice,SCOOTSY LOGISTICS PRIVATE LIMITED,1.051890e+18,CUS-00167,1.051890e+18,Branch-Maharashtra,...,"Waredepot, C/O Top India Logistics, Gala No. 1...",NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Select required columns and rename

In [15]:
import pandas as pd

# 1️⃣ Rename columns
df = df.rename(columns={
    'PurchaseOrder': 'PO Number',
    'Billing Address': 'invoice_bill_to',
    'Shipping Address': 'invoice_ship_to',
    'CF.Dispatch From': 'invoice_dispatch_from',
    'Item Name': 'Invoice Item & Description',
    'Quantity': 'Invoice_Quantity',
    'Item Total': 'Invoice_Amount',
    'Item Price': 'Invoice Rate'
})

# 2️⃣ Split Item Tax % into SGST % and CGST %
df['Item Tax %'] = pd.to_numeric(df['Item Tax %'], errors='coerce')
df['invoice_amt_gst'] = df['Invoice_Amount'] + df['Item Tax Amount']
df['SGST %'] = df['Item Tax %'] / 2
df['CGST %'] = df['Item Tax %'] / 2

# 3️⃣ Split Item Tax Amount into SGST Amt and CGST Amt
df['Item Tax Amount'] = pd.to_numeric(df['Item Tax Amount'], errors='coerce')

df['SGST Amt'] = df['Item Tax Amount'] / 2
df['CGST Amt'] = df['Item Tax Amount'] / 2

# 4️⃣ Create blank IGST columns
df['IGST %'] = None
df['IGST Amt'] = None

# 5️⃣ Drop original tax columns
df = df.drop(columns=['Item Tax %', 'Item Tax Amount'])

# 6️⃣ Keep only required columns
required_columns = [
    'Invoice Number',
    'PO Number',
    'Invoice Date',
    'SKU',
    'invoice_bill_to',
    'invoice_ship_to',
    'invoice_dispatch_from',
    'Invoice Item & Description',
    'Invoice_Quantity',
    'Invoice Rate',
    'Invoice_Amount',
    'invoice_amt_gst',
    'SGST %',
    'CGST %',
    'IGST %',
    'SGST Amt',
    'CGST Amt',
    'IGST Amt'
]

df = df[required_columns]


In [16]:
df

,Invoice Number,PO Number,Invoice Date,SKU,invoice_bill_to,invoice_ship_to,invoice_dispatch_from,Invoice Item & Description,Invoice_Quantity,Invoice Rate,Invoice_Amount,invoice_amt_gst,SGST %,CGST %,IGST %,SGST Amt,CGST Amt,IGST Amt
547,IN/2025-26/001,MBIPO298320,2025-12-10,ZB-CBP12RO10UB,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...","Compostable Bagasse Plates, 12 inch-Round, 10 ...",90.0,119.952,10795.68,11335.46,2.5,2.5,None,269.890,269.890,None
548,IN/2025-26/001,MBIPO298320,2025-12-10,ZB-CBB6RO25UB,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Bagasse Bowls Round 180 ml count o...,64.0,62.333,3989.31,4188.77,2.5,2.5,None,99.730,99.730,None
549,IN/2025-26/001,MBIPO298320,2025-12-10,ZB-CNB5CMT10UB.,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Bagasse - 5 Compartment Meal Tray ...,80.0,119.952,9596.16,10075.96,2.5,2.5,None,239.900,239.900,None
550,IN/2025-26/001,MBIPO298320,2025-12-10,ZB-CRC5OZ25NL,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Ripple Cups 5 Oz - Set of 25 - NL,80.0,64.788,5183.04,6115.98,9.0,9.0,None,466.470,466.470,None
551,IN/2025-26/001,MBIPO298320,2025-12-10,ZB-CBP9RO10UB.,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Bagasse plates 9 Inch Set of 10,100.0,78.048,7804.80,8195.04,2.5,2.5,None,195.120,195.120,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151018,IN/2025-26/064,GCAPO35386,2026-01-27,ZB-CPN2P50P01,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,"Waredepot, C/O Top India Logistics, Gala No. 1...",Compostable Paper Napkin 2 Ply - Set Of 50 - P...,50.0,41.483,2074.15,2447.50,9.0,9.0,None,186.675,186.675,None
151019,IN/2025-26/064,GCAPO35386,2026-01-27,ZB-CBP11RO4C10UB,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,"Waredepot, C/O Top India Logistics, Gala No. 1...","Compostable Bagasse Plates, 11 inch-4 Compartm...",36.0,94.762,3411.43,3582.00,2.5,2.5,None,85.285,85.285,None
151020,IN/2025-26/064,GCAPO35386,2026-01-27,ZB-CPHC8OZ25NL,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,"Waredepot, C/O Top India Logistics, Gala No. 1...",8 Oz/ 250 ml Paper Cup - - Count of 25- Multic...,40.0,60.127,2405.08,2837.99,9.0,9.0,None,216.455,216.455,None
151021,IN/2025-26/064,GCAPO35386,2026-01-27,ZB-OBGR16OZ10,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,"Waredepot, C/O Top India Logistics, Gala No. 1...","Oxo-Biodegradable Beer Pong Glasses, Red, 16 O...",40.0,59.025,2361.00,2785.98,9.0,9.0,None,212.490,212.490,None


In [17]:
import regex as re
def clean_sku(s):
    """
    Clean SKU number by removing prefixes, spaces, and special characters.
    """
    if not s:
        return ""

    s = str(s).upper()

    # Remove leading prefixes (ZB, SP etc.)
    s = re.sub(r"^(ZB|SP)[\s\-\.\_]+", "", s)

    # Remove spaces/hyphens/dots
    s = re.sub(r"[\s\-\.\_]", "", s)

    # Allow only A–Z and 0–9
    s = re.sub(r"[^A-Z0-9]", "", s)

    return s.strip()


# Apply cleaning to SKU column
if 'SKU' in df.columns:
    print(f"Cleaning SKU column...")
    print(f"Before: Sample SKUs = {df['SKU'].head(10).tolist()}")

    # Create a new column with cleaned SKU or keep original if cleaning results in empty
    df['SKU Cleaned'] = df['SKU'].apply(clean_sku)

    # Replace original SKU with cleaned version (or keep original if cleaned is empty)
    df['SKU'] = df.apply(
        lambda row: row['SKU Cleaned'] if row['SKU Cleaned'] else row['SKU'],
        axis=1
    )

    # Drop the temporary cleaned column
    df = df.drop(columns=['SKU Cleaned'])

    print(f"After: Sample SKUs = {df['SKU'].head(10).tolist()}")
    print(f"✅ SKU column cleaned successfully!")
else:
    print("⚠️ 'SKU' column not found in dataframe")

df


Cleaning SKU column...
Before: Sample SKUs = ['ZB-CBP12RO10UB', 'ZB-CBB6RO25UB', 'ZB-CNB5CMT10UB.', 'ZB-CRC5OZ25NL', 'ZB-CBP9RO10UB.', 'ZB-CBP6RO25UB.', 'ZB-PCL5OZ50NL', 'SP-PLP10SQ10', 'ZB-CBDN1P100P01', 'ZB-CPHC5OZ50NL']
After: Sample SKUs = ['CBP12RO10UB', 'CBB6RO25UB', 'CNB5CMT10UB', 'CRC5OZ25NL', 'CBP9RO10UB', 'CBP6RO25UB', 'PCL5OZ50NL', 'PLP10SQ10', 'CBDN1P100P01', 'CPHC5OZ50NL']
✅ SKU column cleaned successfully!


,Invoice Number,PO Number,Invoice Date,SKU,invoice_bill_to,invoice_ship_to,invoice_dispatch_from,Invoice Item & Description,Invoice_Quantity,Invoice Rate,Invoice_Amount,invoice_amt_gst,SGST %,CGST %,IGST %,SGST Amt,CGST Amt,IGST Amt
547,IN/2025-26/001,MBIPO298320,2025-12-10,CBP12RO10UB,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...","Compostable Bagasse Plates, 12 inch-Round, 10 ...",90.0,119.952,10795.68,11335.46,2.5,2.5,None,269.890,269.890,None
548,IN/2025-26/001,MBIPO298320,2025-12-10,CBB6RO25UB,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Bagasse Bowls Round 180 ml count o...,64.0,62.333,3989.31,4188.77,2.5,2.5,None,99.730,99.730,None
549,IN/2025-26/001,MBIPO298320,2025-12-10,CNB5CMT10UB,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Bagasse - 5 Compartment Meal Tray ...,80.0,119.952,9596.16,10075.96,2.5,2.5,None,239.900,239.900,None
550,IN/2025-26/001,MBIPO298320,2025-12-10,CRC5OZ25NL,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Ripple Cups 5 Oz - Set of 25 - NL,80.0,64.788,5183.04,6115.98,9.0,9.0,None,466.470,466.470,None
551,IN/2025-26/001,MBIPO298320,2025-12-10,CBP9RO10UB,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey no 9...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Bagasse plates 9 Inch Set of 10,100.0,78.048,7804.80,8195.04,2.5,2.5,None,195.120,195.120,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151018,IN/2025-26/064,GCAPO35386,2026-01-27,CPN2P50P01,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,"Waredepot, C/O Top India Logistics, Gala No. 1...",Compostable Paper Napkin 2 Ply - Set Of 50 - P...,50.0,41.483,2074.15,2447.50,9.0,9.0,None,186.675,186.675,None
151019,IN/2025-26/064,GCAPO35386,2026-01-27,CBP11RO4C10UB,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,"Waredepot, C/O Top India Logistics, Gala No. 1...","Compostable Bagasse Plates, 11 inch-4 Compartm...",36.0,94.762,3411.43,3582.00,2.5,2.5,None,85.285,85.285,None
151020,IN/2025-26/064,GCAPO35386,2026-01-27,CPHC8OZ25NL,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,"Waredepot, C/O Top India Logistics, Gala No. 1...",8 Oz/ 250 ml Paper Cup - - Count of 25- Multic...,40.0,60.127,2405.08,2837.99,9.0,9.0,None,216.455,216.455,None
151021,IN/2025-26/064,GCAPO35386,2026-01-27,OBGR16OZ10,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,SCOOTSY LOGISTICS PRIVATE LIMITED\nSurvey Nos....,"Waredepot, C/O Top India Logistics, Gala No. 1...","Oxo-Biodegradable Beer Pong Glasses, Red, 16 O...",40.0,59.025,2361.00,2785.98,9.0,9.0,None,212.490,212.490,None


In [18]:
is_unique = ~df.duplicated(subset=["SKU", "PO Number"]).any()
print("Is (SKU + PO Number) unique?", is_unique)

Is (SKU + PO Number) unique? False


In [19]:
df.drop_duplicates(subset=['SKU', 'PO Number'], inplace=True)


In [20]:
invoice_df_month= df.copy()

# PO Invoice merge

In [21]:
merged_df = df_po_month.merge(
    invoice_df_month,
    on=['PO Number', 'SKU'],
    how='left'   # use 'inner' if you want only matched records
)


In [22]:
final_df=merged_df.copy()

# Add platform and channel

In [23]:
final_df[['ClientID', 'Platform', 'Channel']] = [
    'TH-1755734939046', 'QuickCommerce', 'InstaMart']

In [24]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO_quantity,...,invoice_amt_gst,SGST %,CGST %,IGST %,SGST Amt,CGST Amt,IGST Amt,ClientID,Platform,Channel
0,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,203624,"Eco Soul 180ml Round, Disposable Bowl 25.0 Pieces",51.00,326.40,128,...,6776.85,2.5,2.5,None,161.355,161.355,None,TH-1755734939046,QuickCommerce,InstaMart
1,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,284873,"Eco Soul 9 Round, Eco Soul Disposable Plate 10...",70.95,532.14,150,...,9092.63,2.5,2.5,None,216.490,216.490,None,TH-1755734939046,QuickCommerce,InstaMart
2,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,394372,"Eco Soul 1Ply, 100Count, Disposable Paper Napk...",41.48,2240.08,300,...,15407.14,9.0,9.0,None,1175.120,1175.120,None,TH-1755734939046,QuickCommerce,InstaMart
3,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,428727,"Eco Soul 140mm, Disposable Spoon 50.0 Pieces",47.14,707.14,300,...,13581.23,2.5,2.5,None,323.365,323.365,None,TH-1755734939046,QuickCommerce,InstaMart
4,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,756550,"Eco Soul 12 Round, Eco Soul Palm Leaf Plate 10...",135.11,0.00,150,...,21807.75,0.0,0.0,None,0.000,0.000,None,TH-1755734939046,QuickCommerce,InstaMart
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,944680,"Eco Soul 11Round,Eco Soul 4CP Disposable Plate...",94.76,170.57,36,...,3582.00,2.5,2.5,None,85.285,85.285,None,TH-1755734939046,QuickCommerce,InstaMart
1764,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,946960,"Eco Soul 6 Square, Eco Soul Palm Leaf Plate 10...",76.11,0.00,20,...,1522.20,0.0,0.0,None,0.000,0.000,None,TH-1755734939046,QuickCommerce,InstaMart
1765,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,963761,"Eco Soul 250ml, Multi Colour Paper Cup 25.0 Pi...",60.13,432.92,40,...,2837.99,9.0,9.0,None,216.455,216.455,None,TH-1755734939046,QuickCommerce,InstaMart
1766,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,973525,Eco Soul Beer Pong Glasses | Red | 16 Oz 1.0 Pack,59.03,424.98,40,...,2785.98,9.0,9.0,None,212.490,212.490,None,TH-1755734939046,QuickCommerce,InstaMart


# Extract GRN

In [25]:
import os
import pandas as pd

folder_path = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\Quick_Commerece_Input_ALL\Instamart\Month_wise_GRN"

dfs = []

for file in os.listdir(folder_path):
    if file.lower().endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        print(f"📥 Reading GRN file: {file}")
        temp_df = pd.read_csv(file_path)
        dfs.append(temp_df)

grn_df = pd.concat(dfs, ignore_index=True)

print(f"✅ Combined GRN DataFrame shape: {grn_df.shape}")


📥 Reading GRN file: GRN_of_60_days_till_2june.csv
📥 Reading GRN file: GRN_of_60_days_till_4Feb.csv
📥 Reading GRN file: GRN_OF_LAST_30DAYS_TILL_11FEB.csv
📥 Reading GRN file: GRN_OF_LAST_30DAYS_TILL_7MAY.csv
📥 Reading GRN file: GRN_OF_LAST_60DAYS_TILL_9JUNE.csv
📥 Reading GRN file: LAST 60 DAYS TILL 25 MAY 2026.csv
📥 Reading GRN file: LAST_30_DAYS_TILL_16FEB.csv
📥 Reading GRN file: LAST_30_DAYS_TILL_19MARCH.csv
📥 Reading GRN file: LAST_30_DAYS_TILL_9MARCH.csv
📥 Reading GRN file: LAST_60-DAYS_TILL_11MAY.csv
📥 Reading GRN file: LAST_60-DAYS_TILL_20 MAY.csv
📥 Reading GRN file: LAST_60-DAYS_TILL_24APRIL.csv
📥 Reading GRN file: LAST_60-DAYS_TILL_24FEB.csv
📥 Reading GRN file: LAST_60-DAYS_TILL_26FEB.csv
📥 Reading GRN file: LAST_60-DAYS_TILL_29APRIL.csv
📥 Reading GRN file: LAST_60-DAYS_TILL_2march.csv
📥 Reading GRN file: LAST_60DAYS_TILL_23FEB.csv
📥 Reading GRN file: LAST_60DAYS_TILL_6MARCH.csv
📥 Reading GRN file: last_60_days_till_19FEB.csv
✅ Combined GRN DataFrame shape: (8210, 31)


In [26]:
# Remove duplicates based on PO + SKU
grn_final_df = grn_df.drop_duplicates(
    subset=["PurchaseOrderNumber", "SkuCode"],
    keep="first"
).reset_index(drop=True)

print(f"✅ Rows after GRN deduplication: {grn_final_df.shape[0]}")


✅ Rows after GRN deduplication: 1749


# Rename GRN

In [27]:
grn_df = grn_df.rename(columns={
    'PurchaseOrderNumber': 'PO Number',
    'SkuCode': 'grn_item_code',
    'SkuDescription' :'grn_product_description',
    'LotMrp': 'grn_mrp',
    'ReceivedQty' : 'grn_quantity' ,
    'GrnLineValueWithTax': 'grn_total_amount',
    'TotalTax': 'grn_tax_amount',
    'CreatedAtDate': 'dn_date',
    'FacilityName': 'facility_name',
    'DnNumber': 'dn_id',
    'Category': 'item_name',
    'SupplierCode' : 'vendor_invoice_id',
    'DNQuantity' : 'dn_quantity',
    'DNValue' : 'dn_value'
    
    
})
grn_df['dn_reason'] = ""
grn_df['dn_remark'] = ""
grn_df['grn_gmv_loss'] = ""

required_columns = [
    'PO Number',
    'grn_item_code',
    'grn_product_description',
    'grn_mrp',
    'grn_tax_amount',
    'grn_quantity',
    'grn_total_amount',
    'grn_gmv_loss',
    'dn_date',
    'facility_name',
    'dn_id',
    'item_name',
    'vendor_invoice_id',
    'dn_quantity',
    'dn_value',
    'dn_reason',
    'dn_remark'
]
grn_df =grn_df[required_columns]

# Add SKU in grn

In [28]:
import pandas as pd

helper_file = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\helper_file.xlsx"

df_helper = pd.read_excel(helper_file)

# Keep all required columns from helper
cols_to_use = [
    'Item Code',
    'SKU',
]
grn_df["grn_item_code"] = grn_df["grn_item_code"].astype(str).str.strip()
df_helper["Item Code"] = df_helper["Item Code"].astype(str).str.strip()

grn_df = grn_df.merge(
    df_helper[cols_to_use],
    how='left',
    left_on='grn_item_code',
    right_on='Item Code',
    suffixes=('', '_helper')   # left: no suffix, right: _helper
)


# Drop extra Item Code column if not needed
grn_df = grn_df.drop(columns=['Item Code'])

# ---- Find PO Item Codes whose SKU is missing in helper ----
missing_sku = grn_df[grn_df['SKU'].isna()][['grn_item_code']].drop_duplicates()
grn_df.drop(columns='SKU_helper', inplace=True, errors='ignore')
# If you also want to export this list:


# Merge PO, Invoice, GRN

In [29]:
is_unique = ~grn_df.duplicated(subset=["SKU", "PO Number"]).any()
print("Is (SKU + PO_No) unique?", is_unique)

Is (SKU + PO_No) unique? False


In [30]:
duplicate_count = grn_df.duplicated(subset=['PO Number', 'SKU']).sum()
print("Number of duplicate rows:", duplicate_count)


Number of duplicate rows: 7033


In [31]:
grn_df.drop_duplicates(subset=['SKU', 'PO Number'], inplace=True)
grn_df['PO Number']=grn_df['PO Number'].astype(str)


In [32]:
final_df = final_df.merge(
    grn_df,
    on=['PO Number', 'SKU'],
    how='left'
)


In [33]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO_quantity,...,grn_gmv_loss,dn_date,facility_name,dn_id,item_name,vendor_invoice_id,dn_quantity,dn_value,dn_reason,dn_remark
0,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,203624,"Eco Soul 180ml Round, Disposable Bowl 25.0 Pieces",51.00,326.40,128,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,284873,"Eco Soul 9 Round, Eco Soul Disposable Plate 10...",70.95,532.14,150,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,394372,"Eco Soul 1Ply, 100Count, Disposable Paper Napk...",41.48,2240.08,300,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,428727,"Eco Soul 140mm, Disposable Spoon 50.0 Pieces",47.14,707.14,300,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,756550,"Eco Soul 12 Round, Eco Soul Palm Leaf Plate 10...",135.11,0.00,150,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,944680,"Eco Soul 11Round,Eco Soul 4CP Disposable Plate...",94.76,170.57,36,...,,2026-02-11 18:54:54,Viz IM1,NaN,Non Food,71211564.0,0.0,0.0,,
1764,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,946960,"Eco Soul 6 Square, Eco Soul Palm Leaf Plate 10...",76.11,0.00,20,...,,2026-02-11 18:54:54,Viz IM1,NaN,Non Food,71211564.0,0.0,0.0,,
1765,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,963761,"Eco Soul 250ml, Multi Colour Paper Cup 25.0 Pi...",60.13,432.92,40,...,,2026-02-11 18:54:54,Viz IM1,NaN,Non Food,71211564.0,0.0,0.0,,
1766,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,973525,Eco Soul Beer Pong Glasses | Red | 16 Oz 1.0 Pack,59.03,424.98,40,...,,2026-02-11 18:54:54,Viz IM1,NaN,Non Food,71211564.0,0.0,0.0,,


# Add NLC

In [34]:
import pandas as pd

revenue_df = pd.read_excel(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Shambhavi's File\NLC All platform1.xlsx",sheet_name='Instamart',header=1)

# Rename columns
revenue_df = revenue_df.rename(columns={
    'SKUs': 'SKU',
    'MRP':'MRP as per pricing sheet',
    'GST':'GST as per pricing sheet',
})

# (Optional but recommended) keep only required columns
revenue_df = revenue_df[['SKU', 'NLC as per pricing sheet','MRP as per pricing sheet','GST as per pricing sheet']]

# Clean SKU
revenue_df['SKU'] = revenue_df['SKU'].astype(str).str.strip()

final_df = final_df.merge(
    revenue_df[['SKU', 'NLC as per pricing sheet','MRP as per pricing sheet','GST as per pricing sheet']],
    on='SKU',
    how='left'
)
missing_skus = final_df.loc[final_df['NLC as per pricing sheet'].isna(), 'SKU'].unique().tolist()
    

# Add revenue

In [35]:
import numpy as np

# Ensure numeric types (very important)
final_df['NLC as per pricing sheet'] = pd.to_numeric(final_df['NLC as per pricing sheet'], errors='coerce')
final_df['Invoice_Quantity'] = pd.to_numeric(final_df['Invoice_Quantity'], errors='coerce')

# Create revenue column
final_df['revenue'] = final_df['NLC as per pricing sheet'] * final_df['Invoice_Quantity']


# Add status from file

In [36]:
import pandas as pd

# Load instamart file
instamart_file = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Harshit Raut's files - INSTA\ESH (SCM)- INSTAMART Delivery Details.xlsx"

instamart_df = pd.read_excel(instamart_file, sheet_name="Appt Date")


# Select only required columns and rename them
instamart_selected = instamart_df[[
    'PO No', 
    'Appt Date', 
    'Status', 
    'Requested Appt Date'
]].copy()

# Rename columns as requested
instamart_selected = instamart_selected.rename(columns={
    'PO No': 'PO Number',
    'Appt Date': 'appt date', 
    'Status': 'PO_fulfilled_status_from_file',
    'Requested Appt Date': 'requested_appt_date'
})
# Convert PO Number to string in both dataframes
final_df['PO Number'] = final_df['PO Number'].astype(str).str.strip()
instamart_selected['PO Number'] = instamart_selected['PO Number'].astype(str).str.strip()

# Merge with your existing final_df (left join on PO_Number)
final_df = final_df.merge(
    instamart_selected, 
    on='PO Number', 
    how='left'  # Keeps all rows from final_df
)

print("✅ Merge complete!")
print(f"Shape before: {len(final_df)} rows")
print(f"New columns added: appt date, PO_fulfilled_status_from_file, requested_appt_date")


✅ Merge complete!
Shape before: 1768 rows
New columns added: appt date, PO_fulfilled_status_from_file, requested_appt_date


c:\Users\Shubham Verma\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


# Add a status

In [37]:
import numpy as np
final_df["Invoice_Quantity"] = (
    final_df["Invoice_Quantity"]
    .astype(str)
    .str.split("\n")          # split multiline cell
    .str[0]                   # take first value (1120.0)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
)

final_df['PO_quantity'] = pd.to_numeric(
    final_df['PO_quantity'], errors='coerce'
).fillna(0)

final_df['Status'] = np.where(
    final_df['Invoice Number'].isna(),
    'Unfulfilled',
    np.where(final_df['Invoice_Quantity'] < final_df['PO_quantity'], 'Short Shipped', 'Fulfilled')
)


In [38]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO_quantity,...,dn_reason,dn_remark,NLC as per pricing sheet,MRP as per pricing sheet,GST as per pricing sheet,revenue,appt date,PO_fulfilled_status_from_file,requested_appt_date,Status
0,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,203624,"Eco Soul 180ml Round, Disposable Bowl 25.0 Pieces",51.00,326.40,128,...,NaN,NaN,51.00000,119,0.05,6528.0000,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled
1,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,284873,"Eco Soul 9 Round, Eco Soul Disposable Plate 10...",70.95,532.14,150,...,NaN,NaN,70.95238,149,0.05,10642.8570,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled
2,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,394372,"Eco Soul 1Ply, 100Count, Disposable Paper Napk...",41.48,2240.08,300,...,NaN,NaN,41.48305,89,0.18,12444.9150,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled
3,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,428727,"Eco Soul 140mm, Disposable Spoon 50.0 Pieces",47.14,707.14,300,...,NaN,NaN,47.14286,99,0.05,14142.8580,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled
4,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,756550,"Eco Soul 12 Round, Eco Soul Palm Leaf Plate 10...",135.11,0.00,150,...,NaN,NaN,135.11000,229,0.00,20266.5000,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,944680,"Eco Soul 11Round,Eco Soul 4CP Disposable Plate...",94.76,170.57,36,...,,,94.76190,199,0.05,3411.4284,2026-02-11 00:00:00,Fulfilled,2026-02-11 00:00:00,Fulfilled
1764,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,946960,"Eco Soul 6 Square, Eco Soul Palm Leaf Plate 10...",76.11,0.00,20,...,,,76.11000,129,0.00,1522.2000,2026-02-11 00:00:00,Fulfilled,2026-02-11 00:00:00,Fulfilled
1765,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,963761,"Eco Soul 250ml, Multi Colour Paper Cup 25.0 Pi...",60.13,432.92,40,...,,,60.12712,129,0.18,2405.0848,2026-02-11 00:00:00,Fulfilled,2026-02-11 00:00:00,Fulfilled
1766,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,973525,Eco Soul Beer Pong Glasses | Red | 16 Oz 1.0 Pack,59.03,424.98,40,...,,,59.03000,199,0.18,2361.2000,2026-02-11 00:00:00,Fulfilled,2026-02-11 00:00:00,Fulfilled


# Change datatypes

In [39]:
import pandas as pd

# ---------- TEXT columns ----------
text_cols = [
    'PO Number', 'PO Creation Date', 'PO Expiry Date',
    'PO Delivered By Address', 'PO Item Code', 'HSN Code',
    'PO Product UPC', 'PO Product Description',
    'PO Basic Cost Price', 'PO Landing Rate',
    'Invoice Number', 'Invoice Date', 'SKU',
    'Invoice Item & Description', 'Category', 'Material',
    'Sub Category', 'Status', 'ClientID', 'Platform',
    'Channel', 'PO MRP', 'Invoice Rate',
    'CGST %', 'SGST %', 'IGST %',
    'grn_product_description', 'grn_tax_amount',
    'grn_total_amount', 'grn_gmv_loss',
    'dn_date', 'facility_name', 'dn_id',
    'item_name', 'vendor_invoice_id',
    'dn_reason', 'dn_remark',
    'appt date', 'PO_fulfilled_status_from_file',
    'requested_appt_date'
]

for col in text_cols:
    final_df[col] = final_df[col].astype('string')


# ---------- BIGINT(20) columns ----------
bigint_cols = [
    'PO_quantity',
    'Box/Case',
    'grn_item_code',
    'grn_mrp',
    'grn_quantity',
    'MRP as per pricing sheet'
]

for col in bigint_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce').astype('Int64')


# ---------- DOUBLE columns ----------
double_cols = [
    'PO Tax Amt',
    'Invoice_Quantity',
    'PO_Amount',
    'Invoice_Amount',
    'invoice_amt_gst',
    'CGST Amt',
    'SGST Amt',
    'IGST Amt',
    'dn_quantity',
    'dn_value',
    'NLC as per pricing sheet',
    'GST as per pricing sheet',
    'revenue'
]

for col in double_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce').astype('float64')


# Add year, month

In [40]:
final_df['PO Creation Date'] = pd.to_datetime(
    final_df['PO Creation Date'],
    errors='coerce'
)
final_df['year_month'] = final_df['PO Creation Date'].dt.to_period('M').astype(str)
final_df['PO Creation Date'] = final_df['PO Creation Date'].dt.date
float_cols = final_df.select_dtypes(include='float')

final_df[float_cols.columns] = final_df[float_cols.columns].round(2)
import pandas as pd

final_df['year_month'] = pd.to_datetime(
    final_df['year_month'],
    errors='coerce'
)

final_df['year'] = final_df['year_month'].dt.year
final_df['month'] = final_df['year_month'].dt.strftime('%B')
# Convert columns to string type
final_df['year'] = final_df['year'].astype(str)
final_df['month'] = final_df['month'].astype(str)
final_df['year_month'] = final_df['year_month'].astype(str)


In [41]:
final_df['invoice_amt_gst'].sum()

np.float64(7380631.93)

In [42]:
final_df['PO_Amount'].sum()

np.float64(10681964.78)

In [43]:
final_df['year'] = final_df['year'].astype(str)
final_df['month'] = final_df['month'].astype(str).str.zfill(2)

final_df['year_month'] = (
    final_df['year'] + '_' + final_df['month']
)


In [44]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO_quantity,...,MRP as per pricing sheet,GST as per pricing sheet,revenue,appt date,PO_fulfilled_status_from_file,requested_appt_date,Status,year_month,year,month
0,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,203624,"Eco Soul 180ml Round, Disposable Bowl 25.0 Pieces",51.0,326.40,128,...,119,0.05,6528.00,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled,2025_December,2025,December
1,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,284873,"Eco Soul 9 Round, Eco Soul Disposable Plate 10...",70.95,532.14,150,...,149,0.05,10642.86,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled,2025_December,2025,December
2,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,394372,"Eco Soul 1Ply, 100Count, Disposable Paper Napk...",41.48,2240.08,300,...,89,0.18,12444.92,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled,2025_December,2025,December
3,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,428727,"Eco Soul 140mm, Disposable Spoon 50.0 Pieces",47.14,707.14,300,...,99,0.05,14142.86,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled,2025_December,2025,December
4,CADPO167840,2025-12-06,2025-12-28,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,756550,"Eco Soul 12 Round, Eco Soul Palm Leaf Plate 10...",135.11,0.00,150,...,229,0.00,20266.50,2025-12-27 00:00:00,EXPIRED,2025-12-27 00:00:00,Fulfilled,2025_December,2025,December
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,944680,"Eco Soul 11Round,Eco Soul 4CP Disposable Plate...",94.76,170.57,36,...,199,0.05,3411.43,2026-02-11 00:00:00,Fulfilled,2026-02-11 00:00:00,Fulfilled,2026_January,2026,January
1764,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,946960,"Eco Soul 6 Square, Eco Soul Palm Leaf Plate 10...",76.11,0.00,20,...,129,0.00,1522.20,2026-02-11 00:00:00,Fulfilled,2026-02-11 00:00:00,Fulfilled,2026_January,2026,January
1765,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,963761,"Eco Soul 250ml, Multi Colour Paper Cup 25.0 Pi...",60.13,432.92,40,...,129,0.18,2405.08,2026-02-11 00:00:00,Fulfilled,2026-02-11 00:00:00,Fulfilled,2026_January,2026,January
1766,VIAPO38982,2026-01-20,2026-02-12,ECOSOUL HOME PRIVATE LIMITED,SCOOTSY LOGISTICS PRIVATE LIMITED,973525,Eco Soul Beer Pong Glasses | Red | 16 Oz 1.0 Pack,59.03,424.98,40,...,199,0.18,2361.20,2026-02-11 00:00:00,Fulfilled,2026-02-11 00:00:00,Fulfilled,2026_January,2026,January


In [45]:
final_df['PO_fulfilled_status_from_file'].unique()

<StringArray>
[              'EXPIRED',             'Fulfilled',       'Did not deliver',
   'Cancelled by Vendor', 'Appointment Scheduled',          'REFERENCE PO',
            'In Transit']
Length: 7, dtype: string

In [46]:
final_df = final_df[~final_df['PO_fulfilled_status_from_file']
                    .isin(['Did not deliver', 'Cancelled by Vendor','REFERENCE PO'])]


In [47]:
final_df.to_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\instamart_output.csv",index=False)

In [48]:
import pandas as pd
import urllib.parse
import re
from sqlalchemy import create_engine, text, inspect

# ======================================================
# MYSQL CONFIG
# ======================================================
MYSQL_CONFIG = {
    "host": "192.168.50.148",
    "port": 3306,
    "user": "datahive_esh_test",
    "password": "Datahive@321!",
    "database": "datahive_esh_test",
}

DEPARTMENT = "quick_comm"
BASE_TABLE_NAME = "instamart_orders"
TABLE_NAME = f"{DEPARTMENT}_{BASE_TABLE_NAME}"

# ======================================================
# CLEAN MYSQL COLUMN NAMES
# ======================================================
def clean_mysql_column(col):
    col = col.strip()
    col = col.replace(" ", "_")
    col = col.replace("%", "pct")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace("\\", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("[", "")
    col = col.replace("]", "")
    col = re.sub(r"__+", "_", col)
    col = re.sub(r"[^A-Za-z0-9_]", "", col)
    return col.lower()

# ======================================================
# CREATE MYSQL ENGINE
# ======================================================
password_encoded = urllib.parse.quote_plus(MYSQL_CONFIG["password"])

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_CONFIG['user']}:{password_encoded}"
    f"@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}",
    pool_pre_ping=True
)

# ======================================================
# TEST MYSQL CONNECTION (OPTIONAL BUT RECOMMENDED)
# ======================================================
def test_mysql_connection():
    print("\n🔍 Testing MySQL connection...")
    try:
        with engine.connect() as conn:
            db = conn.execute(text("SELECT DATABASE();")).fetchone()
            print("✅ Connected to database:", db[0])
    except Exception as e:
        print("❌ Connection failed:", e)
        return False
    return True

# ======================================================
# SAVE final_df TO MYSQL
# ======================================================
def save_final_df_to_mysql(final_df: pd.DataFrame):
    print("\n🟦 Uploading final_df to MySQL...")

    df = final_df.copy()
    print("📊 Rows to upload:", len(df))

    # Clean column names
    df.columns = [clean_mysql_column(c) for c in df.columns]

    inspector = inspect(engine)
    table_exists = inspector.has_table(TABLE_NAME)

    try:
        with engine.begin() as conn:

            if table_exists:
                print("🗑 Truncating existing table...")
                conn.execute(text(f"TRUNCATE TABLE `{TABLE_NAME}`"))
                mode = "append"
            else:
                print("📌 Creating new table...")
                mode = "fail"

            print("⬆ Inserting rows...")
            df.to_sql(
                TABLE_NAME,
                con=conn,
                if_exists=mode,
                index=False,
                chunksize=1000,
                method="multi"
            )

        print("✅ Upload completed successfully!")

    except Exception as e:
        print("❌ Upload failed:", e)

# ======================================================
# RUN
# ======================================================
# final_df MUST already exist in memory

if test_mysql_connection():
    save_final_df_to_mysql(final_df)



🔍 Testing MySQL connection...
✅ Connected to database: datahive_esh_test

🟦 Uploading final_df to MySQL...
📊 Rows to upload: 1404
🗑 Truncating existing table...
⬆ Inserting rows...
✅ Upload completed successfully!


In [49]:
final_df = pd.read_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\instamart_output.csv")

In [50]:
import pandas as pd
import urllib.parse
import re
from sqlalchemy import create_engine, text, inspect

# ======================================================
# MYSQL CONFIG
# ======================================================
MYSQL_CONFIG = {
    "host": "192.168.50.148",
    "port": 3306,
    "user": "datahive_esh_test",
    "password": "Datahive@321!",
    "database": "datahive_esh_test",
}

DEPARTMENT = "quick_comm"
BASE_TABLE_NAME = "instamart_orders"
TABLE_NAME = f"{DEPARTMENT}_{BASE_TABLE_NAME}"

# ======================================================
# CLEAN MYSQL COLUMN NAMES
# ======================================================
def clean_mysql_column(col):
    col = col.strip()
    col = col.replace(" ", "_")
    col = col.replace("%", "pct")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace("\\", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("[", "")
    col = col.replace("]", "")
    col = re.sub(r"__+", "_", col)
    col = re.sub(r"[^A-Za-z0-9_]", "", col)
    return col.lower()

# ======================================================
# CREATE MYSQL ENGINE
# ======================================================
password_encoded = urllib.parse.quote_plus(MYSQL_CONFIG["password"])

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_CONFIG['user']}:{password_encoded}"
    f"@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}",
    pool_pre_ping=True
)

# ======================================================
# TEST MYSQL CONNECTION (OPTIONAL BUT RECOMMENDED)
# ======================================================
def test_mysql_connection():
    print("\n🔍 Testing MySQL connection...")
    try:
        with engine.connect() as conn:
            db = conn.execute(text("SELECT DATABASE();")).fetchone()
            print("✅ Connected to database:", db[0])
    except Exception as e:
        print("❌ Connection failed:", e)
        return False
    return True

# ======================================================
# SAVE final_df TO MYSQL
# ======================================================
def save_final_df_to_mysql(final_df: pd.DataFrame):
    print("\n🟦 Uploading final_df to MySQL...")

    df = final_df.copy()
    print("📊 Rows to upload:", len(df))

    # Clean column names
    df.columns = [clean_mysql_column(c) for c in df.columns]

    inspector = inspect(engine)
    table_exists = inspector.has_table(TABLE_NAME)

    try:
        with engine.begin() as conn:

            if table_exists:
                print("🗑 Truncating existing table...")
                conn.execute(text(f"TRUNCATE TABLE `{TABLE_NAME}`"))
                mode = "append"
            else:
                print("📌 Creating new table...")
                mode = "fail"

            print("⬆ Inserting rows...")
            df.to_sql(
                TABLE_NAME,
                con=conn,
                if_exists=mode,
                index=False,
                chunksize=1000,
                method="multi"
            )

        print("✅ Upload completed successfully!")

    except Exception as e:
        print("❌ Upload failed:", e)

# ======================================================
# RUN
# ======================================================
# final_df MUST already exist in memory

if test_mysql_connection():
    save_final_df_to_mysql(final_df)



🔍 Testing MySQL connection...


✅ Connected to database: datahive_esh_test

🟦 Uploading final_df to MySQL...
📊 Rows to upload: 1404
🗑 Truncating existing table...
⬆ Inserting rows...
✅ Upload completed successfully!
